# 技巧: 理解模型中的单位

## 目的
在科学计算和工程建模中，不一致的单位是导致错误的最常见原因之一。本框架中的不同模块（如水文模块和水力模块）基于不同的理论，其内部使用的变量单位也不同。清晰地了解每个变量的单位是正确使用本框架的前提。

本教程旨在作为一个快速参考指南，列出框架中关键变量的单位，并演示一个最常见的单位转换场景。

## 1. 核心模块单位参考表

| 模块 | 变量 | 单位 | 描述 |
| :--- | :--- | :--- | :--- |
| **`hydro_model`** | | | **水文模型 (概念性)** |
| | `rainfall` | mm/timestep | 每个时间步的降雨深度 |
| | `pet` | mm/timestep | 每个时间步的潜在蒸散发深度 |
| | `S`, `Q_s` | mm | 内部蓄水状态，以等效水深表示 |
| | `outflow` | mm/timestep | 每个时间步的出流量，以等效水深表示 |
| **`preissmann_model`**| | | **一维水力模型** |
| | `length`, `width` | m | 几何参数：米 |
| | `Z`, `Z_bed` | m | 水位和河床高程：米 |
| | `Q` | m³/s | 流量：立方米每秒 |
| | `dt` | s | 时间步长：秒 |
| **`model_2d`** | | | **二维水力模型** |
| | `h`, `z_bed` | m | 水深和河床高程：米 |
| | `uh`, `vh` | m²/s | x和y方向的单位宽度流量（动量）|
| | `dt` | s | 时间步长：秒 |

## 2. 关键场景: 水文-水力耦合的单位转换

最常见的需要单位转换的场景，是将一个`HydrologicalModel`的输出（单位为`mm/timestep`）作为`HydraulicModel`的输入（单位为`m³/s`）。

这个转换需要两个额外的信息：
1.  **流域面积 (Catchment Area)**: 用于将水深（mm）转换成体积（m³）。
2.  **时间步长 (Time Step)**: 用于将每个时间步的总量转换成每秒的流量。

转换公式如下：

$$ Q \, (m^3/s) = \frac{\text{Runoff}\,(mm)}{1000\,(mm/m)} \times \text{Area}\,(km^2) \times 10^6\,(m^2/km^2) \div \text{dt}\,(s) $$

下面是一个代码示例。

In [ ]:
def convert_runoff_to_discharge(runoff_mm, area_km2, dt_seconds):
    """将以mm为单位的径流深度转换为以m³/s为单位的流量。"""
    
    # 毫米转米
    runoff_m = runoff_mm / 1000.0
    
    # 平方公里转平方米
    area_m2 = area_km2 * 1e6
    
    # 计算总体积
    volume_m3 = runoff_m * area_m2
    
    # 体积除以时间得到流量
    discharge_m3s = volume_m3 / dt_seconds
    
    return discharge_m3s

# --- 示例 ---
# 假设一个日步长(dt=24*3600s)的水文模型，在一个150km²的流域上，
# 计算出当天的总径流深为 5.5 mm。
runoff_depth_mm = 5.5
catchment_area_km2 = 150.0
dt_in_seconds = 24 * 3600

inflow_for_hydraulic_model = convert_runoff_to_discharge(
    runoff_depth_mm,
    catchment_area_km2,
    dt_in_seconds
)

print(f"{runoff_depth_mm} mm/day 的径流，在 {catchment_area_km2} km² 的流域上，")
print(f"相当于 {inflow_for_hydraulic_model:.2f} m³/s 的平均流量。")

## 3. 结论

在进行模型耦合或数据准备时，务必注意单位的统一。
- **水文模型**内部使用`mm`作为基本单位，因为它与降雨量的单位一致，非常直观。
- **水力模型**使用`m`和`s`作为基本单位（米-秒制），因为它们是物理方程（如圣维南方程）的标准单位。

在将它们连接在一起时，上文演示的单位转换是保证模型物理意义正确的关键一步。在`EnKF`的例子中，`model_forward_augmented`函数内部就执行了这样的转换。